In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_final = pd.read_csv("../Práctica 1/glp1_limpio.csv", parse_dates=['FECHA'])

In [2]:
import requests
import pandas as pd

BASE_URL = "https://opendata.nhsbsa.net/api/3/action/datastore_search_sql"

def consulta(sql, mes):
    """Corre una consulta SQL contra la tabla EPD_SNOMED_<mes> y regresa un DataFrame."""
    r = requests.get(BASE_URL,
                     params={"resource_id": f"EPD_SNOMED_{mes}", "sql": sql},
                     timeout=180)
    r.raise_for_status()
    resultado = r.json()["result"]
    registros = resultado.get("records") or resultado.get("result", {}).get("records", [])
    return pd.DataFrame(registros)

In [3]:
MES = "202605"
muestra = consulta(f"SELECT * FROM `EPD_SNOMED_{MES}` LIMIT 1", MES)
print(list(muestra.columns))

['YEAR_MONTH', 'REGIONAL_OFFICE_NAME', 'REGIONAL_OFFICE_CODE', 'ICB_NAME', 'ICB_CODE', 'PCO_NAME', 'PCO_CODE', 'PRACTICE_NAME', 'PRACTICE_CODE', 'ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4', 'POSTCODE', 'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_CHEMICAL_SUBSTANCE', 'BNF_PRESENTATION_CODE', 'BNF_PRESENTATION_NAME', 'BNF_CHAPTER_PLUS_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'UNIDENTIFIED', 'SNOMED_CODE']


In [5]:
COLS_UBICACION = ['YEAR_MONTH', 'REGIONAL_OFFICE_NAME', 'REGIONAL_OFFICE_CODE', 'ICB_NAME', 'ICB_CODE', 'PCO_NAME', 'PCO_CODE', 'PRACTICE_NAME', 'PRACTICE_CODE', 'ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4', 'POSTCODE', 'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_CHEMICAL_SUBSTANCE', 'BNF_PRESENTATION_CODE', 'BNF_PRESENTATION_NAME', 'BNF_CHAPTER_PLUS_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'UNIDENTIFIED', 'SNOMED_CODE'] 

for col in COLS_UBICACION:
    d = consulta(f"SELECT COUNT(DISTINCT {col}) AS n FROM `EPD_SNOMED_{MES}`", MES)
    print(col, "->", d["n"].iloc[0])

YEAR_MONTH -> 1
REGIONAL_OFFICE_NAME -> 8
REGIONAL_OFFICE_CODE -> 8
ICB_NAME -> 37
ICB_CODE -> 37
PCO_NAME -> 374
PCO_CODE -> 375
PRACTICE_NAME -> 8758
PRACTICE_CODE -> 9278
ADDRESS_1 -> 7469
ADDRESS_2 -> 6689
ADDRESS_3 -> 2258
ADDRESS_4 -> 368
POSTCODE -> 7059
BNF_CHEMICAL_SUBSTANCE_CODE -> 1784
BNF_CHEMICAL_SUBSTANCE -> 1635
BNF_PRESENTATION_CODE -> 21275
BNF_PRESENTATION_NAME -> 21274
BNF_CHAPTER_PLUS_CODE -> 21
QUANTITY -> 1914
ITEMS -> 1393
TOTAL_QUANTITY -> 5140
ADQ_USAGE -> 9146
NIC -> 57353
ACTUAL_COST -> 590927
UNIDENTIFIED -> 2
SNOMED_CODE -> 34330


In [6]:
REGION_COL = "REGIONAL_OFFICE_NAME"
CONSULTORIO_COL = "PRACTICE_CODE"
SUSTANCIA_COL = "BNF_CHEMICAL_SUBSTANCE"

TODAS = ["Dulaglutide", "Semaglutide", "Tirzepatide", "Liraglutide", "Exenatide", "Lixisenatide"]

def revenue_por_region(mes, sustancias):
    lista = ", ".join(f"'{s}'" for s in sustancias)
    sql = f"""
    SELECT {REGION_COL} AS region,
           COUNT(DISTINCT {CONSULTORIO_COL}) AS consultorios,
           SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue,
           SUM(SAFE_CAST(ITEMS AS FLOAT64)) AS recetas
    FROM `EPD_SNOMED_{mes}`
    WHERE {SUSTANCIA_COL} IN ({lista})
    GROUP BY region
    """
    return consulta(sql, mes)

glp1 = revenue_por_region(MES, TODAS)
tirz = revenue_por_region(MES, ["Tirzepatide"])

print(glp1.sort_values("revenue", ascending=False))
print(tirz.sort_values("revenue", ascending=False))

                     region  consultorios       revenue   recetas
2                  MIDLANDS          1287  1.838980e+07  103734.0
4                SOUTH EAST           801  1.536028e+07   80114.0
5  NORTH EAST AND YORKSHIRE           959  1.351324e+07   81832.0
6                    LONDON          1182  1.090422e+07   54923.0
0                NORTH WEST           995  1.073977e+07   66583.0
1           EAST OF ENGLAND           671  9.972386e+06   56898.0
3                SOUTH WEST           527  8.827412e+06   47561.0
7              UNIDENTIFIED             1  8.021914e+03      30.0
                     region  consultorios       revenue  recetas
1                  MIDLANDS          1267  1.525082e+07  66175.0
4                SOUTH EAST           789  1.314339e+07  54937.0
5  NORTH EAST AND YORKSHIRE           946  1.124511e+07  52672.0
3                    LONDON          1166  9.098502e+06  35177.0
0           EAST OF ENGLAND           663  8.368985e+06  36918.0
2               

In [ ]:
def armar_tabla(df):
    d = df[df["region"] != "UNIDENTIFIED"].copy()

    d["pct_revenue"] = d["revenue"] / d["revenue"].sum() * 100
    d["pct_consultorios"] = d["consultorios"] / d["consultorios"].sum() * 100
    d["revenue_por_consultorio"] = d["revenue"] / d["consultorios"]

    return d.sort_values("revenue", ascending=False).round(1)

tabla_glp1 = armar_tabla(glp1)
tabla_tirz = armar_tabla(tirz)

print(tabla_glp1)
print(tabla_tirz)


                     region  consultorios     revenue   recetas  pct_revenue  \
2                  MIDLANDS          1287  18389798.4  103734.0         21.0   
4                SOUTH EAST           801  15360276.3   80114.0         17.5   
5  NORTH EAST AND YORKSHIRE           959  13513244.3   81832.0         15.4   
6                    LONDON          1182  10904220.8   54923.0         12.4   
0                NORTH WEST           995  10739769.1   66583.0         12.2   
1           EAST OF ENGLAND           671   9972385.5   56898.0         11.4   
3                SOUTH WEST           527   8827411.7   47561.0         10.1   

   pct_consultorios  revenue_por_consultorio  
2              20.0                  14288.9  
4              12.5                  19176.4  
5              14.9                  14091.0  
6              18.4                   9225.2  
0              15.5                  10793.7  
1              10.4                  14862.0  
3               8.2           

In [8]:
def revenue_por_consultorio(mes, sustancias):
    lista = ", ".join(f"'{s}'" for s in sustancias)
    sql = f"""
    SELECT PRACTICE_CODE AS consultorio,
           SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue
    FROM `EPD_SNOMED_{mes}`
    WHERE BNF_CHEMICAL_SUBSTANCE IN ({lista})
    GROUP BY consultorio
    """
    return consulta(sql, mes)

cons_glp1 = revenue_por_consultorio(MES, TODAS)
cons_tirz = revenue_por_consultorio(MES, ["Tirzepatide"])
print(len(cons_glp1), len(cons_tirz))


6423 6333


In [11]:
def concentracion(df):
    d = df.sort_values("revenue", ascending=False).reset_index(drop=True)
    d["pct_consultorios_acum"] = (d.index + 1) / len(d) * 100
    d["pct_revenue_acum"] = d["revenue"].cumsum() / d["revenue"].sum() * 100
    return d

c_glp1 = concentracion(cons_glp1)
c_tirz = concentracion(cons_tirz)

# % del revenue que genera el 10 % de consultorios con más revenue
print(c_glp1[c_glp1["pct_consultorios_acum"] <= 20]["pct_revenue_acum"].max())
print(c_tirz[c_tirz["pct_consultorios_acum"] <= 20]["pct_revenue_acum"].max())

48.67117103205797
50.555682280252654


In [12]:
rev_tirz = tirz["revenue"].sum()
rev_glp1 = glp1["revenue"].sum()
peso_tirz = rev_tirz / rev_glp1 * 100
print(f"Tirzepatide es el {peso_tirz:.1f} % del revenue de GLP-1 en mayo de 2026")

Tirzepatide es el 83.2 % del revenue de GLP-1 en mayo de 2026


In [13]:
for mes in ["202609", "202610"]:
    try:
        r = consulta(f"""
            SELECT SUM(SAFE_CAST(TOTAL_QUANTITY AS FLOAT64)) AS unidades,
                   SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue
            FROM `EPD_SNOMED_{mes}`
            WHERE BNF_CHEMICAL_SUBSTANCE = 'Tirzepatide'
        """, mes)
        print(mes, r)
    except Exception as e:
        print(mes, "no disponible todavía:", e)

202609 no disponible todavía: 404 Client Error: NOT FOUND for url: https://opendata.nhsbsa.net/api/3/action/datastore_search_sql?resource_id=EPD_SNOMED_202609&sql=%0A++++++++++++SELECT+SUM%28SAFE_CAST%28TOTAL_QUANTITY+AS+FLOAT64%29%29+AS+unidades%2C%0A+++++++++++++++++++SUM%28SAFE_CAST%28ACTUAL_COST+AS+FLOAT64%29%29+AS+revenue%0A++++++++++++FROM+%60EPD_SNOMED_202609%60%0A++++++++++++WHERE+BNF_CHEMICAL_SUBSTANCE+%3D+%27Tirzepatide%27%0A++++++++
202610 no disponible todavía: 404 Client Error: NOT FOUND for url: https://opendata.nhsbsa.net/api/3/action/datastore_search_sql?resource_id=EPD_SNOMED_202610&sql=%0A++++++++++++SELECT+SUM%28SAFE_CAST%28TOTAL_QUANTITY+AS+FLOAT64%29%29+AS+unidades%2C%0A+++++++++++++++++++SUM%28SAFE_CAST%28ACTUAL_COST+AS+FLOAT64%29%29+AS+revenue%0A++++++++++++FROM+%60EPD_SNOMED_202610%60%0A++++++++++++WHERE+BNF_CHEMICAL_SUBSTANCE+%3D+%27Tirzepatide%27%0A++++++++


In [14]:
SEIS = ["Dulaglutide", "Semaglutide", "Tirzepatide", "Liraglutide", "Exenatide", "Lixisenatide"]

def exacto_mes(mes):
    col_sust = "CHEMICAL_SUBSTANCE_BNF_DESCR" if mes < "202503" else "BNF_CHEMICAL_SUBSTANCE"
    lista = ", ".join(f"'{s}'" for s in SEIS)
    sql = f"""
    SELECT PRACTICE_CODE AS consultorio,
           SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue,
           SUM(SAFE_CAST(ITEMS AS FLOAT64)) AS items,
           SUM(IF({col_sust} = 'Tirzepatide', SAFE_CAST(ACTUAL_COST AS FLOAT64), 0)) AS revenue_tirz
    FROM `EPD_SNOMED_{mes}`
    WHERE {col_sust} IN ({lista})
    GROUP BY consultorio
    """
    return consulta(sql, mes)

meses = [f"{a}{m:02d}" for a in (2024, 2025) for m in range(1, 13)]
partes = []
for mes in meses:
    d = exacto_mes(mes)
    d["mes"] = mes
    partes.append(d)
    print(mes, len(d), "consultorios")

api_2425 = pd.concat(partes)
exacto = api_2425.groupby("consultorio")[["revenue", "items", "revenue_tirz"]].sum().reset_index()
print(len(exacto), exacto["revenue"].sum())

202401 6426 consultorios
202402 6429 consultorios
202403 6445 consultorios
202404 6430 consultorios
202405 6414 consultorios
202406 6426 consultorios
202407 6424 consultorios
202408 6408 consultorios
202409 6379 consultorios
202410 6379 consultorios
202411 6390 consultorios
202412 6380 consultorios
202501 6383 consultorios
202502 6384 consultorios
202503 6396 consultorios
202504 6406 consultorios
202505 6433 consultorios
202506 6406 consultorios
202507 6415 consultorios
202508 6388 consultorios
202509 6422 consultorios
202510 6404 consultorios
202511 6394 consultorios
202512 6395 consultorios
7159 892348429.12997


In [15]:
peso_recetas = tirz["recetas"].sum() / glp1["recetas"].sum() * 100
print(f"Tirzepatide es el {peso_recetas:.1f} % de las recetas de GLP-1 en mayo de 2026")

Tirzepatide es el 64.1 % de las recetas de GLP-1 en mayo de 2026


In [16]:
# 1b. Meses extra: el costo por receta original usa todo el periodo
meses_extra = [f"2026{m:02d}" for m in range(1, 6)]
partes_extra = []
for mes in meses_extra:
    d = exacto_mes(mes)
    d["mes"] = mes
    partes_extra.append(d)
    print(mes, len(d), "consultorios")

api_todo = pd.concat([api_2425] + partes_extra)   # de 2024-01 a 2026-05, datos exactos

202601 6390 consultorios
202602 6398 consultorios
202603 6418 consultorios
202604 6427 consultorios
202605 6423 consultorios


In [18]:
api_todo.to_csv("../Práctica 7/api_2024_2026_por_consultorio.csv", index=False)

In [21]:
meses_ant = [f"{a}{m:02d}" for a in (2021, 2022, 2023) for m in range(1, 13)]
partes_ant = []
for mes in meses_ant:
    d = exacto_mes(mes)
    d["mes"] = mes
    partes_ant.append(d)
    print(mes, len(d), "consultorios")

api_completo = pd.concat(partes_ant + [api_todo])
api_completo.to_csv("../Práctica 7/api_2021_2026_por_consultorio.csv", index=False)
print(len(api_completo))

202101 6603 consultorios
202102 6593 consultorios
202103 6610 consultorios
202104 6627 consultorios
202105 6598 consultorios
202106 6603 consultorios
202107 6597 consultorios
202108 6588 consultorios
202109 6588 consultorios
202110 6583 consultorios
202111 6576 consultorios
202112 6567 consultorios
202201 6565 consultorios
202202 6569 consultorios
202203 6588 consultorios
202204 6576 consultorios
202205 6589 consultorios
202206 6573 consultorios
202207 6553 consultorios
202208 6559 consultorios
202209 6556 consultorios
202210 6541 consultorios
202211 6513 consultorios
202212 6509 consultorios
202301 6520 consultorios
202302 6518 consultorios
202303 6527 consultorios
202304 6507 consultorios
202305 6530 consultorios
202306 6510 consultorios
202307 6490 consultorios
202308 6453 consultorios
202309 6436 consultorios
202310 6438 consultorios
202311 6378 consultorios
202312 6386 consultorios
421329


In [22]:
meses_fy = [f"2025{m:02d}" for m in range(4, 13)] + [f"2026{m:02d}" for m in range(1, 4)]
rev_fy, items_fy = 0, 0
for mes in meses_fy:
    d = consulta(f"""
        SELECT SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue,
               SUM(SAFE_CAST(ITEMS AS FLOAT64)) AS items
        FROM `EPD_SNOMED_{mes}`
        WHERE BNF_CHEMICAL_SUBSTANCE = 'Tirzepatide'
    """, mes)
    rev_fy += float(d["revenue"].iloc[0])
    items_fy += float(d["items"].iloc[0])

print(f"Tirzepatide 2025/26 -> revenue: £{rev_fy/1e6:.0f} M | items: {items_fy/1e6:.2f} M")

Tirzepatide 2025/26 -> revenue: £570 M | items: 3.06 M


In [23]:
costo_tirz = tirz["revenue"].sum() / tirz["recetas"].sum()
costo_resto = (glp1["revenue"].sum() - tirz["revenue"].sum()) / (glp1["recetas"].sum() - tirz["recetas"].sum())
print(round(costo_tirz, 1), round(costo_resto, 1), round(costo_tirz / costo_resto, 2))

231.8 83.2 2.79


In [26]:
TODAS = ["Dulaglutide", "Semaglutide", "Tirzepatide", "Liraglutide", "Exenatide", "Lixisenatide"]
lista = ", ".join(f"'{s}'" for s in TODAS)
import numpy as np
from scipy.stats import kruskal

def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n

print("Gini GLP-1:", round(gini(cons_glp1["revenue"]), 3))
print("Gini Tirzepatide:", round(gini(cons_tirz["revenue"]), 3))

d = consulta(f"""
    SELECT PRACTICE_CODE AS consultorio, REGIONAL_OFFICE_NAME AS region,
           SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue
    FROM `EPD_SNOMED_202605`
    WHERE BNF_CHEMICAL_SUBSTANCE IN ({lista})
    GROUP BY consultorio, region
""", "202605")
d["revenue"] = d["revenue"].astype(float)
d = d[d["region"] != "UNIDENTIFIED"]
grupos = [g["revenue"].values for _, g in d.groupby("region")]
H, p = kruskal(*grupos)
print("H =", round(H, 1), "| p =", p, "| epsilon² =", round(H / (len(d) - 1), 3))

Gini GLP-1: 0.452
Gini Tirzepatide: 0.475
H = 456.3 | p = 2.1880133620606853e-95 | epsilon² = 0.071


In [27]:
import pandas as pd

meses = [f"{a}{m:02d}" for a in range(2021, 2027) for m in range(1, 13) if f"{a}{m:02d}" <= "202605"]
filas = []
for mes in meses:
    col = "BNF_CHEMICAL_SUBSTANCE" if mes >= "202503" else "CHEMICAL_SUBSTANCE_BNF_DESCR"
    d = consulta(f"""
        SELECT SUM(SAFE_CAST(ACTUAL_COST AS FLOAT64)) AS revenue_total,
               SUM(IF({col} = 'Tirzepatide', SAFE_CAST(ACTUAL_COST AS FLOAT64), 0)) AS revenue_tirz
        FROM `EPD_SNOMED_{mes}`
        WHERE {col} IN ({lista})
    """, mes)
    d["mes"] = mes
    filas.append(d)

mensual = pd.concat(filas, ignore_index=True)
mensual[["revenue_total", "revenue_tirz"]] = mensual[["revenue_total", "revenue_tirz"]].astype(float)
mensual["anio"] = mensual["mes"].str[:4]

anual_x = mensual.groupby("anio")[["revenue_total", "revenue_tirz"]].sum() / 1e6
anual_x["otros_glp1"] = anual_x["revenue_total"] - anual_x["revenue_tirz"]
print(anual_x.round(1))

      revenue_total  revenue_tirz  otros_glp1
anio                                         
2021          167.7           0.0       167.7
2022          219.4           0.0       219.4
2023          237.9           0.0       237.9
2024          286.0          70.1       215.9
2025          606.4         417.0       189.3
2026          420.1         346.7        73.4
